# SimpleAudit Analysis

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os, warnings
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import analyze_data as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11

print('Ready.')

## Load Data

In [ ]:
DATA_ROOT = 'comparison_results/norwegian_safety'

# ── Aggregate (run-level) data ─────────────────────────────────────────────
df_raw = ad.load_comparison_results(DATA_ROOT)
print(f'Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns.')
print(f'Safety columns: target_safe={df_raw.target_safe.value_counts().to_dict()}')

# ── Scenario-level data ────────────────────────────────────────────────────
df_scen = ad.load_scenario_results(DATA_ROOT)
print(f'Scenario rows: {len(df_scen)}')

df_raw.head(3)

## Explore Available Combinations

Size keys marked with `*` are unsafe model variants.  
Use this to know which values to pass to `fixed={}` in analysis functions.

In [ ]:
ad.show_available_combos(df_raw)

## Judges
### Note:
* XL as judge model has only been used for the safe targets. So all judge analyses with reference to XL are limited to safe targets
* The below tells us that judges < M are usless, hence they will be excluded for all further analysis
* L is from here on considered a reliable judge

In [ ]:
#filter out toy runs before analyzing (would never use an S or XS auditor)
df_scen2= ad.filter_by_size(df_scen, "auditor",["M","L", "XL"])
fig = ad.plot_critical_miss_summary2(df_scen2, reference='XL', n_bootstrap=1000, show_title=False)
plt.savefig('img/fig_judge_agreement_full.png', bbox_inches='tight')
plt.show()

In [ ]:
df_scen2= ad.filter_by_size(df_scen, "auditor",["M","L", "XL"])
df_scen2 = ad.filter_by_size(df_scen2, "judge",["XS","M", "L", "XL"])
fig = ad.plot_critical_miss_summary2(df_scen2, reference='XL', n_bootstrap=10, show_title=False)
plt.savefig('img/fig_judge_agreement2.png', bbox_inches='tight')
plt.show()

In [ ]:
#df is difficult to read with errors, hence bootstraping is deactivated
rates = ad.compute_critical_miss_rates(df_scen, reference='XL', n_bootstrap=0)
print('Agreement vs J=XL:')
display(rates.round(3))

In [ ]:
display(rates[['judge','downgrade_rate', 'upgrade_rate']].round(3))

In [ ]:
# Choose L or M as judge?
# Using J=XL as the reference (ground truth), compare L and M pairwise.

merged_L = ad.compute_severity_confusion(df_scen, judge_a='XL', judge_b='L')
merged_M = ad.compute_severity_confusion(df_scen, judge_a='XL', judge_b='M')

# ── 1. Headline rates ──────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Judge': ['L', 'M'],
    'Exact agreement': [merged_L['agreement'].mean(), merged_M['agreement'].mean()],
    'Upgrade rate (judge > XL)': [(merged_L['ordinal_diff'] > 0).mean(),
                                   (merged_M['ordinal_diff'] > 0).mean()],
    'Downgrade rate (judge < XL)': [(merged_L['ordinal_diff'] < 0).mean(),
                                     (merged_M['ordinal_diff'] < 0).mean()],
}).set_index('Judge')

# ── 2. Critical downgrade: when XL says critical, how often does each miss? ─
xl_crit_L = merged_L[merged_L['sev_a'] == 'critical']
xl_crit_M = merged_M[merged_M['sev_a'] == 'critical']
summary['Miss rate on XL=critical'] = [
    (xl_crit_L['ordinal_diff'] < 0).mean(),
    (xl_crit_M['ordinal_diff'] < 0).mean(),
]
summary['n (XL=critical)'] = [len(xl_crit_L), len(xl_crit_M)]

print('=== Overall calibration vs J=XL ===')
display(summary.round(3))

# ── 3. Mean ordinal bias by XL severity level ──────────────────────────────
bias_L = merged_L.groupby('sev_a')['ordinal_diff'].mean().rename('L')
bias_M = merged_M.groupby('sev_a')['ordinal_diff'].mean().rename('M')
bias = pd.concat([bias_L, bias_M], axis=1).reindex(['pass','low','medium','high','critical'])
bias.index.name = 'XL severity'
print('\n=== Mean ordinal bias per XL level  (positive = judge rates higher) ===')
display(bias.round(3))


# print(bias.round(3).to_markdown())


## Filtering

In [ ]:
# ── filter_by_safety: specify any combination of role safety ───────────────

# Only rows where the target was an unsafe model (the canonical comparison set)
df_unsafe_targets = ad.filter_by_safety(df_raw, target_safe=False)
print(f'Unsafe targets only: {len(df_unsafe_targets)} rows')



#  all-safe models, strict hierarchy, no small judges
df_safe = (
    df_raw
    .pipe(ad.filter_all_safe)
    .pipe(ad.filter_strict_hierarchy_judge)
    .pipe(ad.filter_min_judge_size, 'M')
)
print(f'Clean (all-safe, J>T, judge>=M): {len(df_safe)} rows')


# unsafe targets, strict hierarchy, no small judges, safe judges/auditors only
df_unsafe = (
    df_raw
    .pipe(ad.filter_by_safety, target_safe=False, judge_safe=True, auditor_safe=True)
    .pipe(ad.filter_strict_hierarchy_judge)
    .pipe(ad.filter_min_judge_size, 'M')
)
print(f'Unsafe targets only, but safe judges/auditors: {len(df_unsafe)} rows')


## Default df for analysis
# safe and unsafe targets, strict hierarchy, no small judges, safe judges/auditors only, no toy auditors
df = (
    df_raw
    .pipe(ad.filter_semistrict_hierarchy_judge)
    .pipe(ad.filter_min_judge_size, 'M')
    .pipe(ad.filter_by_safety, judge_safe = True, auditor_safe = True)
    .pipe(ad.filter_by_size, "auditor",["M","L", "XL"])
)

df_strict = (
    df_raw
    .pipe(ad.filter_strict_hierarchy_judge)
    .pipe(ad.filter_min_judge_size, 'M')
    .pipe(ad.filter_by_safety, judge_safe = True, auditor_safe = True)
    .pipe(ad.filter_by_size, "auditor",["M","L", "XL"])
)
print(f'Safe and unsafe targets, but safe judges/auditors: {len(df)} rows')


## Variance Decomposition

Partial η² measures the unique contribution of each factor controlling for the others.

| Threshold | Interpretation |
|-----------|---------------|
| ≥ 0.06    | small         |
| ≥ 0.14    | medium        |
| ≥ 0.26    | large         |

Stars: `***` p<0.001 &nbsp; `**` p<0.01 &nbsp; `*` p<0.05 &nbsp; `ns`

In [ ]:


df_temp = ad.filter_by_size(df, "auditor", ["M", "L"])
df_temp = ad.filter_by_size(df_temp, "judge", ["M", "L"])
# df_temp = ad.filter_by_safety(df_temp, target_safe=False)
# df_temp = ad.filter_all_safe(df_temp)

fig = ad.plot_variance_decomposition(df_temp, outcome='score', n_bootstrap=10, show_title=False)
# plt.savefig('img/fig_variance_decomposition_mixed_targets.png', bbox_inches='tight')
plt.show()

# NOTE: the p_value column contains a statsmodels bug.
# Because auditor and judge share the same size labels (M/L/XL), the Type-II ANOVA
# constraint matrix is rank-deficient, causing statsmodels to use df1=3 internally
# for auditor and judge instead of the correct df1=2 (3 levels − 1).
# This makes their p-values appear ~8 and ~3 orders of magnitude too small respectively,
# and breaks monotonicity (p_target > p_auditor despite F_target > F_auditor).
# Corrected p-values (scipy, df1=2, df2=378): target ~5.6e-30, auditor ~4.7e-23, judge ~2.9e-9.
# All three effects remain massively significant; the F statistics and partial n2 are unaffected

In [ ]:
# Underlying numbers
result = ad.compute_variance_decomposition(df_temp, outcome='score', n_bootstrap=1000)
display(result[['factor', 'partial_eta2', 'F', 'p_value', 'n_levels']])
result[['factor', 'partial_eta2', 'p_value', 'n_levels', "eta2_lo", "eta2_hi"]]

In [ ]:
# Severity-level facets: does factor importance differ across severity levels?
fig = ad.plot_variance_decomposition(df_temp, outcome='severity', n_bootstrap=10, show_title=False)
# plt.savefig('img/fig_variance_decomposition_by_severity.png', bbox_inches='tight')
plt.show()

## Score by a Single Axis

In [ ]:
X_AXIS  = 'target'
FIXED   = {'auditor': 'XL', 'judge': 'XL'}  # hold these constant
OUTCOME = 'score'

fig = ad.plot_by_axis(df, x=X_AXIS, fixed=FIXED, outcome=OUTCOME)
# plt.savefig('img/fig_score_by_target.png', bbox_inches='tight')
plt.show()

## Heatmap — Two Factors at Once

Gray cells (`×`) indicate missing data combinations.

In [ ]:
X_AXIS  = 'target'
Y_AXIS  = 'auditor'
FIXED   = {'judge': 'XL'}
OUTCOME = 'score'

fig = ad.plot_heatmap(df_safe, x=X_AXIS, y=Y_AXIS, fixed=FIXED, outcome=OUTCOME, show_title=False)
plt.savefig('img/fig_score_heatmap.png', bbox_inches='tight')
plt.show()

## Severity Profile

Same aggregate score can hide very different severity distributions.
The stacked bar shows the mean count per severity level; the panel below shows the score.

In [ ]:
# GROUP_BY = 'auditor'
# FIXED    = {'target': 'S', 'judge': 'XL'}

# fig = ad.plot_severity_profile(df, group_by=GROUP_BY, fixed=FIXED)
# plt.show()

## Safe vs Unsafe Comparison

In [ ]:
# ── Approach A: convenience wrapper ───────────────────────────────────────
# split_by='target' compares rows with safe vs unsafe targets.
fig = ad.plot_safety_split(
    df,
    x='target',
    split_by='target',
    fixed={'auditor': 'M', 'judge': 'L'},
    outcome='score'
)
# plt.savefig('img/fig_safe_vs_unsafe.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Box + strip: safe vs unsafe score distributions by target size ────────
auditor_plot = 'M'
judge_plot   = 'L' 

_s = df[(df['target_safe'] == True)  & (df['auditor'] == auditor_plot) & (df['judge'] == judge_plot)].copy()
_u = df[(df['target_safe'] == False) & (df['auditor'] == auditor_plot) & (df['judge'] == judge_plot)].copy()
_s['Safety'] = 'Safe'; _u['Safety'] = 'Unsafe'

plot_df = pd.concat([_s, _u])
SIZE_ORDER = ['XS', 'S', 'M']
PALETTE    = {'Safe': '#4C72B0', 'Unsafe': '#DD8452'}
RNG        = np.random.default_rng(42)

fig, axes = plt.subplots(1, 3, figsize=(8, 3.8), sharey=True)

for ax, sz in zip(axes, SIZE_ORDER):
    sub = plot_df[plot_df['target'] == sz]
    for i, grp in enumerate(['Safe', 'Unsafe']):
        vals = sub[sub['Safety'] == grp]['score'].dropna().values
        color = PALETTE[grp]
        ax.boxplot(
            vals, positions=[i], widths=0.42, patch_artist=True,
            medianprops=dict(color='white', linewidth=2.2),
            boxprops=dict(facecolor=color, alpha=0.55),
            whiskerprops=dict(color=color, linewidth=1.3),
            capprops=dict(color=color, linewidth=1.3),
            flierprops=dict(marker=''),
        )
        jitter = RNG.uniform(-0.14, 0.14, len(vals))
        ax.scatter(i + jitter, vals, color=color, s=38, zorder=4,
                   alpha=0.9, edgecolors='white', linewidths=0.5)

    ax.set_title(f'Target = {sz}', fontsize=11)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Safe', 'Unsafe'], fontsize=10)
    ax.set_xlim(-0.6, 1.6)
    ax.grid(axis='y', alpha=0.25, linewidth=0.8)
    ax.set_ylim(-2, 102)

axes[0].set_ylabel('Score', fontsize=10)
# fig.suptitle(
#     f'Score distributions: safe vs unsafe targets  |  auditor={auditor_plot}, judge={judge_plot}',
#     fontsize=11
# )
plt.tight_layout()
plt.savefig('img/fig_safe_unsafe_distributions.png', bbox_inches='tight')
plt.show()

### b) Separation Quality — AUROC

**Why AUROC?**  
The bar chart above shows means and standard deviations, but a single number that captures
*both* the gap and the overlap is more useful for comparing across experiments (different judges,
auditors, or systems such as Petri).

AUROC = **P(score_safe > score_unsafe)** over all random safe–unsafe pairs.
It is computed via the Mann-Whitney U statistic (non-parametric, no normality assumption):

$$\text{AUROC} = \frac{U}{n_{\text{safe}} \times n_{\text{unsafe}}}$$

**How to read it:**
| AUROC | Meaning |
|-------|---------|
| 1.00 | Every safe run scored above every unsafe run — perfect separation |
| 0.95 | In 95% of random safe–unsafe pairs the safe one scored higher |
| 0.50 | Chance — no useful signal |

The bootstrap 95% CI reflects uncertainty from the small n=10 per condition.

Two granularities are reported:
- **Per target size** — the clean comparison: same-size safe vs same-size unsafe.
- **Overall (pooled)** — all safe vs all unsafe regardless of size; confounded by the
  size effect (a safe XS and an unsafe M score similarly), so it is a *conservative* bound.

In [ ]:
from scipy.stats import mannwhitneyu

# ── Mirror the same split as the plot above ────────────────────────────────
# fixed: auditor=M, judge=L; split on target_safe

auditor = 'L'
judge   = 'L'

_safe   = df[(df['target_safe'] == True)  & (df['auditor'] == auditor) & (df['judge'] == judge)]
_unsafe = df[(df['target_safe'] == False) & (df['auditor'] == auditor) & (df['judge'] == judge)]

SIZE_ORDER = ['XS', 'S', 'M']
N_BOOT     = 1000
RNG        = np.random.default_rng(42)

def _auroc_with_ci(a, b, n_boot=N_BOOT, rng=RNG):
    "AUROC = P(a > b), with percentile bootstrap CI."
    stat, _ = mannwhitneyu(a, b, alternative='greater')
    auroc   = stat / (len(a) * len(b))
    boot = [
        mannwhitneyu(
            rng.choice(a, len(a), replace=True),
            rng.choice(b, len(b), replace=True),
            alternative='greater'
        )[0] / (len(a) * len(b))
        for _ in range(n_boot)
    ]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return auroc, lo, hi

rows = []
for sz in SIZE_ORDER:
    s = _safe[_safe['target']   == sz]['score'].dropna().values
    u = _unsafe[_unsafe['target'] == sz]['score'].dropna().values
    if len(s) and len(u):
        auroc, lo, hi = _auroc_with_ci(s, u)
        rows.append({'target': sz, 'n_safe': len(s), 'n_unsafe': len(u),
                     'AUROC': auroc, 'CI_lo': lo, 'CI_hi': hi})

# Overall (pooled across all target sizes)
s_all = _safe['score'].dropna().values
u_all = _unsafe['score'].dropna().values
auroc_all, lo_all, hi_all = _auroc_with_ci(s_all, u_all)
rows.append({'target': 'ALL (pooled)', 'n_safe': len(s_all), 'n_unsafe': len(u_all),
             'AUROC': auroc_all, 'CI_lo': lo_all, 'CI_hi': hi_all})

auroc_df = pd.DataFrame(rows)
auroc_df['95% CI'] = auroc_df.apply(
    lambda r: f"[{r.CI_lo:.3f}, {r.CI_hi:.3f}]", axis=1
)
display(
    auroc_df[['target', 'n_safe', 'n_unsafe', 'AUROC', '95% CI']]
    .rename(columns={'target': 'Target size'})
    .style.format({'AUROC': '{:.3f}'})
    .set_caption(f'AUROC: P(safe score > unsafe score)  |  auditor={auditor}, judge={judge}  |  '
                 f'{N_BOOT:,}-sample bootstrap CI')
)

In [ ]:
print(f"auditor={auditor}, judge={judge}")
print(auroc_df[['target', 'n_safe', 'n_unsafe', 'AUROC', '95% CI']]
    .rename(columns={'target': 'Target size'}))

In [ ]:
# ── Score overlap per target size ─────────────────────────────────────────
# "Overlap zone" = score range where both distributions have members:
#   [max(min_safe, min_unsafe),  min(max_safe, max_unsafe)]
# Empty when max(unsafe) < min(safe) — zero ambiguity.
#
# "Inversions" = pairwise (unsafe_i >= safe_j) — i.e. 1 - AUROC expressed as counts.

rows_ov = []
for sz in SIZE_ORDER:
    s = _safe[_safe['target']     == sz]['score'].dropna().values
    u = _unsafe[_unsafe['target'] == sz]['score'].dropna().values
    if not (len(s) and len(u)):
        continue

    lo_zone = max(s.min(), u.min())
    hi_zone = min(s.max(), u.max())
    zone_exists = lo_zone <= hi_zone

    n_safe_in   = int(((s >= lo_zone) & (s <= hi_zone)).sum()) if zone_exists else 0
    n_unsafe_in = int(((u >= lo_zone) & (u <= hi_zone)).sum()) if zone_exists else 0
    n_in_zone   = n_safe_in + n_unsafe_in
    pct_zone    = 100 * n_in_zone / (len(s) + len(u))

    n_pairs      = len(s) * len(u)
    n_inversions = int(sum((ui >= si) for ui in u for si in s))
    pct_inv      = 100 * n_inversions / n_pairs

    rows_ov.append({
        'Target': sz,
        'Overlap zone': f'[{lo_zone:.1f}, {hi_zone:.1f}]' if zone_exists else '— none —',
        'Safe pts in zone': n_safe_in,
        'Unsafe pts in zone': n_unsafe_in,
        '% of all pts in zone': pct_zone,
        f'Inversions (unsafe≥safe)': f'{n_inversions} / {n_pairs}',
        'Inversion rate %': pct_inv,
    })

ov_df = pd.DataFrame(rows_ov)
display(
    ov_df.style
    .format({'% of all pts in zone': '{:.0f}%', 'Inversion rate %': '{:.1f}%'})
    .hide(axis='index')
    .set_caption('Score overlap  |  auditor=L, judge=L')
)

In [ ]:
# ── Visualise the calibration shift ───────────────────────────────────────
jl     = df[df['judge'] == 'L']
colors = {'M': '#4878CF', 'L': '#6ACC65'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, (title, safe_val) in zip(axes, [('Safe targets', True), ('Unsafe targets', False)]):
    sub   = jl[jl['target_safe'] == safe_val]
    xs    = np.arange(len(SIZE_ORDER))
    width = 0.35

    for i_a, aud in enumerate(['M', 'L']):
        means = []
        stds  = []
        for sz in SIZE_ORDER:
            v = sub[(sub['auditor'] == aud) & (sub['target'] == sz)]['score'].dropna().values
            means.append(v.mean() if len(v) else np.nan)
            stds .append(v.std()  if len(v) else np.nan)

        offsets = xs + (i_a - 0.5) * width
        ax.bar(offsets, [m if not np.isnan(m) else 0 for m in means], width,
               yerr=[s if not np.isnan(s) else 0 for s in stds], capsize=5,
               color=colors[aud], alpha=0.8, label=f'A={aud}',
               error_kw=dict(elinewidth=1.5, ecolor='black'))

        for j, (m, s) in enumerate(zip(means, stds)):
            if not np.isnan(m):
                ax.text(offsets[j], m + (s or 0) + 1.5, f'{m:.1f}',
                        ha='center', va='bottom', fontsize=8)

    ax.set_xticks(xs)
    ax.set_xticklabels([f'T={sz}' for sz in SIZE_ORDER])
    ax.set_title(f'{title}  (J=L)', fontsize=11)
    ax.set_ylabel('Score', fontsize=10)
    ax.set_ylim(0, 85)
    ax.legend()
    ax.yaxis.set_minor_locator(mticker.AutoMinorLocator())
    ax.grid(axis='y', linestyle='--', alpha=0.4)

# fig.suptitle('A=L shifts all scores down relative to A=M — safe and unsafe alike',
#              fontsize=12, y=1.02)
plt.tight_layout()
# plt.savefig('img/fig_auditor_calibration.png', bbox_inches='tight')
plt.show()

## Scenario analysis

In [ ]:
# Apply the same filters to the scenario DataFrame
df_scen_clean = (
    df_scen
    .pipe(ad.filter_all_safe)
    .pipe(ad.filter_strict_hierarchy_judge)
    .pipe(ad.filter_min_judge_size, 'M')
)
print(f'Scenario rows (clean): {len(df_scen_clean)}')
print(f'Scenarios per judge: {df_scen_clean.groupby("judge", observed=True).size().to_dict()}')


In [ ]:
# ── Which scenarios are hardest / most variable? ───────────────────────────
profiles = ad.compute_scenario_profiles(df_scen_clean)
print('Top 10 hardest scenarios (highest mean severity):')
display(profiles.head(10)[['scenario_name', 'mean_ordinal', 'std_ordinal',
                            'modal_severity', 'pct_critical_high', 'pct_pass']])

print('\n10 most consistently passed scenarios:')
display(profiles.tail(10)[['scenario_name', 'mean_ordinal', 'pct_pass']].reset_index(drop=True))


In [ ]:
# Scenario profiles for unsafe targets only — where should auditors catch issues?
profiles_u = ad.compute_scenario_profiles(
    ad.filter_by_safety(df_scen, target_safe=False, judge_safe=True)
)
print('Hardest scenarios for unsafe targets:')
display(profiles_u.head(10)[['scenario_name', 'mean_ordinal', 'pct_critical_high']])